In [4]:
%cd ..
from src.data import load
from src.features import build_features

df = load()
X, y, step, amount = build_features(df)
X.head()

/Users/lakshyashetty/Projects


FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/transactions.parquet'

In [5]:
X.assign(isFraud=y).groupby("isFraud").mean().T

isFraud,0,1
log_amount,11.924932,1.289203e+01
is_transfer,0.191446,4.988433e-01
hour,15.319966,1.154645e+01
is_night,0.009438,2.430293e-01
round_amount,0.001080,3.689273e-02
dest_prev_count,5.301681,1.772190e+00
dest_prev_mean,247186.535774,8.562848e+04
amount_vs_dest_mean,51077.301717,1.001613e+06
steps_since_dest_last,30.018869,3.539851e+01
orig_prev_count,0.000642,1.217582e-03


In [6]:
X.assign(isFraud=y).groupby("isFraud").median().T

isFraud,0,1
log_amount,12.049626,12.997762
is_transfer,0.000000,0.000000
hour,16.000000,12.000000
is_night,0.000000,0.000000
round_amount,0.000000,0.000000
dest_prev_count,3.000000,0.000000
dest_prev_mean,189937.625714,0.000000
amount_vs_dest_mean,1.050026,131597.730000
steps_since_dest_last,9.000000,-1.000000
orig_prev_count,0.000000,0.000000


In [7]:
import numpy as np
big_transfer_new = (X["is_transfer"] == 1) & (X["dest_prev_count"] == 0) & (X["log_amount"] > np.log1p(1_000_000))
night = X["hour"] <= 5

for name, mask in [("All transactions", slice(None)),
                   ("Transfers > $1M to new receivers", big_transfer_new),
                   ("...and at night", big_transfer_new & night)]:
    print(f"{name:35s} count {y[mask].size:>9,} | fraud rate {y[mask].mean():.2%}")

All transactions                    count 2,770,409 | fraud rate 0.30%
Transfers > $1M to new receivers    count    21,202 | fraud rate 6.41%
...and at night                     count       384 | fraud rate 88.80%


In [8]:
new_dest = X["dest_prev_count"] == 0
for label, t in [("CASH_OUT", 0), ("TRANSFER", 1)]:
    m = (X["is_transfer"] == t) & new_dest
    print(f"{label:9s} to a brand-new receiver: count {m.sum():>9,} | fraud rate {y[m].mean():.2%}")

CASH_OUT  to a brand-new receiver: count   426,165 | fraud rate 0.33%
TRANSFER  to a brand-new receiver: count    83,400 | fraud rate 4.91%


In [11]:
import joblib
bundle = joblib.load("models/model.joblib")
model, th = bundle["model"], bundle["threshold"]

m = big_transfer_new & night
p = model.predict_proba(X.loc[m, bundle["features"]])[:, 1]
is_fraud = y[m].to_numpy() == 1
print(f"Real cases: {m.sum()} | median score {np.median(p):.3f} | fraud caught {np.mean(p[is_fraud] >= th):.1%}")

in_train = m & (step <= 300)
print(f"Of these, in the training period: {in_train.sum()} cases, {y[in_train].sum()} frauds")

Real cases: 384 | median score 0.993 | fraud caught 100.0%
Of these, in the training period: 148 cases, 128 frauds


In [10]:
%cd ~/Projects/fraud-detection

/Users/lakshyashetty/Projects/fraud-detection


In [13]:
import pandas as pd
from src.features import features_for_one

empty_state = {"dest": {}, "orig": {}}
base = {"step": 700, "type": "TRANSFER", "nameOrig": "C123456", "nameDest": "C999999999"}

for amt in [2_500_000, 2_500_001, 2_437_189.53]:
    x = pd.DataFrame([features_for_one({**base, "amount": amt}, empty_state)])[bundle["features"]]
    print(f"amount {amt:>14,.2f} -> score {model.predict_proba(x)[0, 1]:.3f}")

print("\nShare of the real 384 cases with a round amount:", X.loc[m, "round_amount"].mean())

amount   2,500,000.00 -> score 0.993
amount   2,500,001.00 -> score 0.993
amount   2,437,189.53 -> score 0.993

Share of the real 384 cases with a round amount: 0.08072916666666667


In [14]:
state = joblib.load("models/state.joblib")
print("In saved history:", state["dest"].get("C999999999"))
print("Times it appears in the data:", (df["nameDest"] == "C999999999").sum())

In saved history: None
Times it appears in the data: 0
